# Qualcomm AI Hub Optimization Workflow for WAFT-Stereo Depth Estimation
This notebook guides you through the process of optimizing and deploying the **WAFT-Stereo** model on edge hardware, specifically the **Dragonwing RB3 Gen 2 Vision Kit**, using the **Qualcomm AI Hub** client library.

## Prerequisite Files:
1. [export_onnx.py](file:///C:/Users/tlamn/Documents/AI/WAFT-Stereo/export_onnx.py): Script that exports the PyTorch WAFT-Stereo model to a standard ONNX format (`ckpts/DAv2S-4.onnx`).
2. [test_onnx_inference.py](file:///C:/Users/tlamn/Documents/AI/WAFT-Stereo/test_onnx_inference.py): A script demonstrating local CPU ONNX Runtime inference using `onnxruntime`.
3. [app.py](file:///C:/Users/tlamn/Documents/AI/WAFT-Stereo/app.py): The main web application which runs depth estimation and interactive 3D point cloud generation using ONNX Runtime.

## Steps Covered:
- **Section 1: ONNX Conversion (Qualcomm cloud)** - Compiling the standard ONNX model (packaged with external weights) into an optimized target format for ONNX Runtime (ORT) on Qualcomm hardware.
- **Section 2: ONNX Runtime Inferences (Dragonwing RB3 Gen 2 Vision Kit)** - Preprocessing and loading real images from `assets/Bottles` to run remote hardware-accelerated inference and performance profiling on a physical device hosted in the Qualcomm cloud.
- **Section 3: Quantization Workflow** - Quantizing the FP32 model to INT8 representation to optimize throughput and memory usage on edge NPUs.

In [ ]:
# Import the Qualcomm AI Hub Python SDK to interact with Qualcomm's cloud compilation, profiling, and quantization services.
# How to use: Ensure 'pip install qai-hub' is executed in your environment, then import it to access client functions.
import qai_hub as hub

# Import numpy for generating mock inputs and handling calibration dataset arrays for quantization.
# How to use: Import numpy as np and utilize array operations like np.random.randn to supply input data formats.
import numpy as np

# Import OpenCV to read, convert, and resize images for feeding the neural network.
# How to use: Import cv2 and use functions like cv2.imread, cv2.cvtColor, cv2.resize to preprocess images.
import cv2

# Import os for operating system interactions, such as creating directories and joining file paths.
# How to use: Import os and use os.makedirs, os.path.exists, etc.
import os

# Import shutil for high-level file operations, such as copying files.
# How to use: Import shutil and call shutil.copy(source, destination) to copy files.
import shutil

# Configure the Qualcomm AI Hub client connection to authenticate API access to cloud devices.
# How to use: Set the QAI_HUB_API_TOKEN environment variable or run 'qai-hub configure --api_token <TOKEN>' in terminal.
# The client automatically uses stored credentials to authorize jobs.
client = hub.Client()

## 1. ONNX Conversion (Qualcomm cloud)
In this section, we package the local base ONNX model structure (`ckpts/DAv2S-4.onnx`) and its external weights data file (`ckpts/DAv2S-4.onnx.data`) generated by [export_onnx.py](file:///C:/Users/tlamn/Documents/AI/WAFT-Stereo/export_onnx.py) into a model directory. We then submit this directory to the Qualcomm cloud compilers to generate an optimized target ONNX representation for ONNX Runtime (ORT) on the target device.

In [ ]:
# Create a dedicated directory to package the ONNX model structure and its external weights file together.
# How to use: Call os.makedirs with the target directory path and set exist_ok=True to prevent error if it already exists.
os.makedirs("ckpts/DAv2S-4_model_dir", exist_ok=True)

# Copy the ONNX model structure file (containing graph metadata) to the packaged model directory.
# How to use: Call shutil.copy, passing the source file path and the destination folder path.
shutil.copy("ckpts/DAv2S-4.onnx", "ckpts/DAv2S-4_model_dir/DAv2S-4.onnx")

# Copy the external weights binary data file to the packaged model directory.
# How to use: Call shutil.copy, passing the source file path and the destination folder path.
shutil.copy("ckpts/DAv2S-4.onnx.data", "ckpts/DAv2S-4_model_dir/DAv2S-4.onnx.data")

# Define the target hardware device (Dragonwing RB3 Gen 2 Vision Kit) for model compilation and optimization.
# How to use: Pass the official device name string to hub.Device() to instantiate a device identifier object.
device = hub.Device("Dragonwing RB3 Gen 2 Vision Kit")

# Define input specifications mapping the input tensor names ('img1', 'img2') to their static dimensions (batch, channels, height, width).
# How to use: Create a dictionary containing the names specified in export_onnx.py and define their shapes as tuple.
input_shapes = {"img1": (1, 3, 480, 640), "img2": (1, 3, 480, 640)}

# Submit a compilation job to Qualcomm cloud compilers to optimize the model specifically for the target device and runtime.
# How to use: Call hub.submit_compile_job, passing the packaged model directory, target device, input specs, and compile options.
# We set options="--target_runtime onnx" to target the ONNX Runtime framework with QNN Execution Provider.
compile_job = hub.submit_compile_job(
    model="ckpts/DAv2S-4_model_dir",
    device=device,
    input_specs=input_shapes,
    options="--target_runtime onnx"
)

# Block current execution and wait for the Qualcomm AI Hub compilation job to finish processing in the cloud.
# How to use: Call get_target_model() on the returned CompileJob object to retrieve the optimized model asset reference.
compiled_model = compile_job.get_target_model()

# Download the compiled, hardware-optimized ONNX model file from the Qualcomm cloud to your local workspace directory.
# How to use: Call download() on the compiled TargetModel object, passing the desired destination file path.
local_compiled_path = compiled_model.download("ckpts/DAv2S-4_compiled.onnx")

## 2. ONNX Runtime Inferences (Dragonwing RB3 Gen 2 Vision Kit)
To verify the numerical correctness of our compiled model and evaluate its performance characteristics on real physical hardware, we load real images (`assets/Bottles/left.png` and `assets/Bottles/right.png`), preprocess them, and submit an inference and profiling job directly to a cloud-hosted Dragonwing RB3 Gen 2 Vision Kit.

In [ ]:
# Read the left camera image file from the assets/Bottles directory in BGR format.
# How to use: Call cv2.imread with the relative path to the image.
left_image_bgr = cv2.imread("assets/Bottles/left.png")

# Read the right camera image file from the assets/Bottles directory in BGR format.
# How to use: Call cv2.imread with the relative path to the image.
right_image_bgr = cv2.imread("assets/Bottles/right.png")

# Define a function to preprocess input images to the format expected by the WAFT-Stereo ONNX model.
# How to use: Define a function that accepts a BGR image array and returns a normalized BCHW float32 numpy array.
def preprocess_image(img_bgr, target_size=(480, 640)):
    # Convert the image color space from BGR (OpenCV default) to RGB (Model expected).
    # How to use: Call cv2.cvtColor passing the image array and cv2.COLOR_BGR2RGB flag.
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Resize the image to match the model's exact expected height and width (480x640).
    # How to use: Call cv2.resize with the image array and (width, height) tuple.
    img_resized = cv2.resize(img_rgb, (target_size[1], target_size[0]))
    
    # Normalize pixel intensity values from range [0, 255] to [0.0, 1.0] by dividing by 255.0.
    # How to use: Convert array to float32 using .astype(np.float32) then divide by 255.0.
    img_float = img_resized.astype(np.float32) / 255.0
    
    # Define ImageNet mean parameters used during training for input normalization.
    # How to use: Define a float32 numpy array with the red, green, and blue mean values.
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    
    # Define ImageNet standard deviation parameters used during training for input normalization.
    # How to use: Define a float32 numpy array with the red, green, and blue std values.
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    
    # Standardize image channels by subtracting the channel mean and dividing by the standard deviation.
    # How to use: Perform element-wise subtraction and division: (image - mean) / std.
    img_norm = (img_float - mean) / std
    
    # Reorder image dimensions from HWC (Height, Width, Channels) to CHW (Channels, Height, Width) layout.
    # How to use: Call transpose(2, 0, 1) to swap the channel dimension to the front.
    img_tensor = img_norm.transpose(2, 0, 1)
    
    # Insert a batch dimension at axis 0 to form a BCHW tensor shape matching the model input shape.
    # How to use: Call np.expand_dims with the tensor array and axis=0.
    img_batch = np.expand_dims(img_tensor, axis=0)
    
    # Return the fully preprocessed float32 input batch.
    # How to use: Return the img_batch array from the function.
    return img_batch

# Preprocess the left image using the defined helper function.
# How to use: Call preprocess_image, passing the BGR left image.
input_left_tensor = preprocess_image(left_image_bgr)

# Preprocess the right image using the defined helper function.
# How to use: Call preprocess_image, passing the BGR right image.
input_right_tensor = preprocess_image(right_image_bgr)

# Assemble preprocessed tensors into a dictionary matching the model's required input names.
# How to use: Wrap each numpy array in a list, as QAI Hub inputs accept multiple evaluation rounds/batches.
inference_inputs = {"img1": [input_left_tensor], "img2": [input_right_tensor]}

# Submit an inference job to execute the compiled model using the real inputs on the physical cloud-hosted device.
# How to use: Call hub.submit_inference_job, specifying the compiled model object, target device, and input dictionary.
inference_job = hub.submit_inference_job(
    model=compiled_model,
    device=device,
    inputs=inference_inputs
)

# Retrieve the inference outputs generated on the physical Dragonwing RB3 Gen 2 Vision Kit device.
# How to use: Call download_output_data() on the completed InferenceJob object to retrieve a dictionary of numpy outputs.
inference_results = inference_job.download_output_data()

# Extract the list of output names (keys) from the downloaded results dictionary.
# How to use: Call list(inference_results.keys()) to get a list of all output keys present in the dictionary.
output_keys = list(inference_results.keys())

# Extract the predicted disparity map from the downloaded outputs using the dynamically retrieved first output key.
# How to use: Retrieve the first key from output_keys (e.g. index 0), then get the first inference run result (index 0).
disp_prediction = inference_results[output_keys[0]][0]

# Submit a profiling job to capture hardware execution metrics such as latency, memory bandwidth, and processor utilization.
# How to use: Call hub.submit_profile_job, specifying the compiled model object and target device.
profile_job = hub.submit_profile_job(
    model=compiled_model,
    device=device
)

# Display a summarized report of the performance metrics returned by the profiling job.
# How to use: Print the result of calling the summarize() method on the completed ProfileJob object.
print(profile_job.summarize())

## 3. Quantization Workflow & Calibration Dataset Preparation Guide
For edge deployment, converting models from float32 (FP32) to 8-bit integer (INT8) representation is critical. This enables execution on the high-efficiency Qualcomm NPU (Neural Processing Unit), significantly reducing latency and memory utilization.

### Calibration Dataset Preparation Guide

To perform Post-Training Quantization (PTQ) successfully without degrading model accuracy, you must prepare a proper calibration dataset. Here are the core guidelines:

1. **Representative Samples**:
   - **Why**: Quantization determines the dynamic range of activations (scale and zero-point parameters) by executing calibration inputs through the network. Using random data or a biased subset will lead to incorrect scaling factors and significant accuracy drop.
   - **How**: Select **100 to 500 representative sample pairs** from your actual validation or training dataset. These should capture the full variance of the deployment environment (e.g. varying lighting, target distances, textures).

2. **Preprocessing Consistency**:
   - **Why**: Calibration data must match the exact numerical distribution the model expects during inference. Any difference in dimensions, color space conversions, image normalization, or axis layout (e.g. BGR vs RGB) will skew calibration scales.
   - **How**: Preprocess each image feed through the exact same pipeline used in [test_onnx_inference.py](file:///C:/Users/tlamn/Documents/AI/WAFT-Stereo/test_onnx_inference.py) and [app.py](file:///C:/Users/tlamn/Documents/AI/WAFT-Stereo/app.py).

3. **Data Format for QAI Hub**:
   Qualcomm AI Hub accepts calibration datasets in two ways:
   - **In-Memory Dictionary Mapping**: A dictionary mapping input layer names (`img1`, `img2`) to lists of preprocessed NumPy arrays (e.g., `{"img1": [array_1, array_2], ...}`). Recommended for small datasets or quick trials.
   - **Uploaded Cloud Dataset**: Creating a persistent dataset asset in Qualcomm AI Hub via `hub.upload_dataset(data, name=...)` or `client.upload_dataset(...)`. This is recommended for production as it allows reusing the dataset across different quantization runs (e.g., comparing INT8 vs INT16 weights) or model architectures without uploading massive files repeatedly. Cloud datasets expire after 30 days.

In [ ]:
# Define a list of real matched left/right stereo image paths from our local assets folder to construct calibration data.
# How to use: Define a list of tuples containing matching left and right image file paths.
calibration_image_paths = [
    ("assets/Bottles/left.png", "assets/Bottles/right.png"),
    ("assets/Keyboard/left.png", "assets/Keyboard/right.png")
]

# Create empty lists to hold preprocessed calibration tensor arrays.
# How to use: Initialize empty python lists for left and right camera inputs.
left_calibration_list = []
right_calibration_list = []

# Iterate over path tuples, load the images, run preprocessing, and append them to calibration lists.
# How to use: Write a standard python for loop over the path list.
for left_path, right_path in calibration_image_paths:
    # Load BGR left image.
    # How to use: Call cv2.imread with the left path.
    left_img = cv2.imread(left_path)
    
    # Load BGR right image.
    # How to use: Call cv2.imread with the right path.
    right_img = cv2.imread(right_path)
    
    # Preprocess and append left image tensor using the utility defined in Section 2.
    # How to use: Call preprocess_image and append the resulting array to left_calibration_list.
    left_calibration_list.append(preprocess_image(left_img))
    
    # Preprocess and append right image tensor using the utility defined in Section 2.
    # How to use: Call preprocess_image and append the resulting array to right_calibration_list.
    right_calibration_list.append(preprocess_image(right_img))

# Group calibration lists in a dictionary mapping model inputs to preprocessed arrays.
# How to use: Create a dictionary matching the schema expected by submit_quantize_job.
calibration_dict = {
    "img1": left_calibration_list,
    "img2": right_calibration_list
}

# Upload the calibration dataset to Qualcomm AI Hub cloud to enable cloud-side quantization.
# How to use: Call hub.upload_dataset (or client.upload_dataset) passing the dictionary and a dataset name string.
calibration_dataset = hub.upload_dataset(
    data=calibration_dict,
    name="waft_stereo_calibration_dataset"
)

# Submit a quantization job to convert the original FP32 ONNX model into a quantized INT8 model on the QAI Hub cloud.
# How to use: Call hub.submit_quantize_job, specifying the packaged model directory (to include external weights),
# the uploaded calibration dataset, and INT8 quantize types.
quantize_job = hub.submit_quantize_job(
    model="ckpts/DAv2S-4_model_dir",
    calibration_data=calibration_dataset,
    weights_dtype=hub.QuantizeDtype.INT8,
    activations_dtype=hub.QuantizeDtype.INT8
)

# Retrieve the quantized model artifact from the completed quantization job.
# How to use: Call get_target_model() on the returned QuantizeJob object to obtain the quantized model reference.
quantized_model = quantize_job.get_target_model()

# Download the quantized ONNX model file locally.
# How to use: Call download() on the quantized Model object to save the resulting Fake-Quantized QDQ ONNX file.
local_quantized_path = quantized_model.download("ckpts/DAv2S-4_quantized.onnx")

# Compile the quantized model for execution on the target device to generate the final device-specific optimized artifact.
# How to use: Submit a compile job with the quantized model, specifying the target device, shapes, and target runtime.
quantized_compile_job = hub.submit_compile_job(
    model=quantized_model,
    device=device,
    input_specs=input_shapes,
    options="--target_runtime onnx"
)

# Retrieve the final compiled quantized model artifact.
# How to use: Call get_target_model() on the completed compile job to reference the optimized INT8 compiled model.
compiled_quantized_model = quantized_compile_job.get_target_model()

# Download the compiled quantized ONNX model locally for integration into edge apps like app.py.
# How to use: Call download() on the compiled quantized TargetModel object.
local_compiled_quant_path = compiled_quantized_model.download("ckpts/DAv2S-4_quantized_compiled.onnx")